<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-01-assistant-configuration/notebook.ipynb)


# Session 1 — Configure the assistant and the repository instructions

**Goal:** turn a generic assistant into a project-aware collaborator, and practice the inspect → plan → edit → test → review loop.

This session runs in your coding assistant, not in this notebook. The notebook is your checklist and logbook.

In [6]:
# manual-run: edits assistant configuration — run with your assistant open
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
CORPUS_DIR = REPO_ROOT / "data" / "corpus"

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    LIVE = preflight(REPO_ROOT)  # the configured lane's client; FakeLLM whenever the lane is down

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [7]:
from bootcamp_agent.checks import check, review

## An assistant that lives in this notebook

Session 1 is about the assistant that edits your files, and that one runs
outside the notebook. This is a different and much smaller thing: a coach that
answers **from the course's own pages**, in this kernel, with no key, no
network, and no second window.

It quotes the page it used. A page id is something you can open; a confident
paraphrase is not.

It only knows the weeks that are in **your** clone, so if you ask about a week
that has not opened yet it will say so rather than invent it.

In [20]:
from bootcamp_agent.coach import coach

# coach("which sessions need a coding assistant?")

# Try your own. These land well today:
# coach("how do I install uv")
# coach("do I need an API key")
# coach("can I use colab")
#
# And one that should REFUSE, because nothing in the pages supports it:
coach("what is the capital of Peru")
#
# It is NOT always right, and it got worse this week. Ask it "what is AGENTS.md
# for?" and the top hit is the bonus page about improving the coach — a page
# added two days ago, which now outranks the pages that actually answer. Adding
# one document degraded retrieval for questions it has nothing to do with.
# That is measurable, and beating it is an open pull request — see the bonus unit.

NOT IN THESE PAGES.
Nothing in your clone shares a word with that question. Either the
course does not cover it, or its week has not been published yet —
`git pull` on a Monday is what brings the next one.


## 1. Warm-up: weak prompt vs project-aware prompt

The same task, asked two ways. **Run this here** — no assistant, no key, no
second window. It uses whichever lane your `.env` names.

- **Weak:** *"add a search feature"*
- **Project-aware:** *"read AGENTS.md, then propose a plan to add a tags filter to `search_documents` in `src/bootcamp_agent/tools.py` — plan only, no edits"*

**What to look for.** The weak prompt has to invent a codebase: it will name
files that do not exist here and pick an architecture nobody asked for. The
project-aware one is bounded — one file, one change, and permission to do
nothing else.

**On the `fake` lane both answers are identical**, because `FakeLLM` returns the
same canned string whatever you ask. That is a correct run and a boring lesson.
For the real contrast, point `.env` at a local model — `make ollama`, then
`BOOTCAMP_PROVIDER=ollama`. See [a local model](../../unit0/local-model.mdx).
It is free and it stays on your machine.

In [21]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED. On `fake` both answers come back identical — that is
# FakeLLM being deterministic, not the prompts being equivalent.
# To stand above it: run it again on a real lane and paste what changed.
# ---------------------------------------------------------------------
from bootcamp_agent.config import load_settings
from bootcamp_agent.llm import get_client

WEAK = "add a search feature"
PROJECT_AWARE = (
    "read AGENTS.md, then propose a plan to add a tags filter to "
    "search_documents in src/bootcamp_agent/tools.py — plan only, no edits"
)

settings = load_settings()
client = get_client(settings)
print(f"lane: {settings.provider}\n")

for label, prompt in (("WEAK", WEAK), ("PROJECT-AWARE", PROJECT_AWARE)):
    print(f"--- {label}")
    print(client.complete(system="You are a careful engineer.", user=prompt)[:600])
    print()

lane: fake

--- WEAK
{"answer": "I do not know based on the provided corpus.", "citations": [], "confidence": 0.0, "needs_human_review": true}

--- PROJECT-AWARE
{"answer": "I do not know based on the provided corpus.", "citations": [], "confidence": 0.0, "needs_human_review": true}



## 2. The task loop, enforced by you

Feature: **add an optional `tags` filter to `search_documents`** — work in a scratch branch/copy.

- [ ] Ask for a **plan** first. Read it. Restrict files it may touch.
- [ ] Ask for the **smallest implementation**.
- [ ] Inspect the **diff** yourself, line by line.
- [ ] Verify with what this repository gives you: `uv run ruff check src/bootcamp_agent/tools.py`,
      then the `check(...)` cell below. (There is no `pytest` here — the suite holds the solved
      value of every exercise and is never published. `AGENTS.md` says so too.)
- [ ] **Reject at least one change** — unsafe, unnecessary, or out of scope — and record what you rejected and why below.
- [ ] Ask for a summary of remaining risks.

**Rejected change + reason:** I rejected changing the retrieval cap. The assignment was only to add an optional `tags` filter in `src/bootcamp_agent/tools.py`; changing the cap would be unrelated and outside the approved scope.

## 4. The loop, applied to this course

From tomorrow on, every exercise ends with a `check(...)` cell. Give your assistant the exercise's **Context** and **Instructions**, let it fill the `TODO(you)` lines, then run the check yourself. You read the verdict, not the assistant.

## 3. Improve the instructions

Where did the assistant assume wrong? That sentence belongs in `AGENTS.md`. Make the edit, note it here — instructions are code (see `docs/guides/harness-engineering.md`).

**Instruction improvement:** The `tags` requirement showed an ambiguity: when several values are supplied, an assistant should not silently choose whether the filter matches any tag or all tags, or assume the input shape. I added a rule to `AGENTS.md` requiring clarification before editing.

## 5. Exercise: the task loop, evidenced

**Context.** The loop only works if you can show you ran it. Four pieces of evidence, and the third one is the whole session: a change you refused. If you rejected nothing, you were not reviewing.

**Instructions.**

1. Work the feature (a `tags` filter on `search_documents`) in a scratch copy, through plan, edit, test, review.
2. Fill each field from what actually happened. `plan_approved` is filled as an example; replace it with yours.
3. Run the check. It refuses a blank rejection, because a loop with no rejection is not the loop.

In [22]:
# ---------------------------------------------------------------------
# THIS RUNS AS SHIPPED, and passes its check. That is the floor.
# But of every cell in this course, THIS is the one where shipped text is
# worth the least: it is a record of somebody else's session, not yours.
# To stand above it: replace all five with what YOUR assistant actually did —
# especially `rejected_change`, which is the whole session.
# ---------------------------------------------------------------------
loop = {
    "plan_approved": "It proposed editing only tools.py: add an optional tags argument, filter before retrieve while preserving the behavious when no tags are supplied, leave the cap alone.",
    "diff_inspected": "Few lines in search_documents. I read the filter placement: it runs before retrieve, so the cap still applies. New optional tags parameter that filters documents with the matching tag, multiple tags can be applied",
    "rejected_change": "It proposed increasing MAX_SEARCH_RESULTS, but I rejected that as it's unrelated",
    "why_rejected": "The only changes should be applied to optional tags.",
    "risks": "It flagged whether the tags should be case sensitive, and what would happen with invalid input such as a plain string instead of a list. and also if multiple tags should match only the documents with all tags provided, or any document that includes any provided tag",
}
for key, value in loop.items():
    print(f"{key:18} {'(empty)' if not value else value[:58]}")

plan_approved      It proposed editing only tools.py: add an optional tags ar
diff_inspected     Six lines in search_documents plus one new test. I read th
rejected_change    It also 'tidied' documents.py, reordering the header parse
why_rejected       Out of scope. An unrequested refactor in a file I was not 
risks              It flagged that an unknown tag silently returns nothing ra


**Expected output** (yours may differ in wording, not in shape):

```
plan_approved      It proposed editing only tools.py: add an optional tags
diff_inspected     Six lines in search_documents plus one new test. I read
...
✅ ch01-e1 passed
```

In [29]:
check("ch01-e1", loop)

✅ ch01-e1 passed


True

## Review

The scorecard for this notebook. Every ❌ line names the exercise and the hint.

In [31]:
review("ch01")

ch01: 1/1 passed  ·  100/100 marks


True

## Exit ticket

Homework: keep the improved instruction file; bring the rejected-change story to tomorrow's warm-up.